# Home Credit — E02 feature-family ablation

Notebook này đo riêng giá trị tăng thêm của E02-A đến E02-E trên cùng baseline E01. Mỗi lượt chạy dùng cùng fold assignment, LightGBM configuration và application train/test. Các experiment chạy tuần tự để giữ mức dùng RAM phù hợp 16 GB.

Không dùng leaderboard để chọn family. Chỉ ghi metric sau khi model thực sự chạy xong.

## Thiết kế experiment

- `E01_locked`: baseline không có feature E02.
- `E02-A_credit_amount`: ratios/amount.
- `E02-B_age_employment`: age/employment.
- `E02-C_external_sources`: external-source availability.
- `E02-D_application_contact`: document/contact/application summaries.
- `E02-E_housing`: housing summaries.
- `E02-ALL`: toàn bộ 18 feature, dùng như consistency check.

Mỗi family được thêm trực tiếp lên E01, không cộng dồn theo thứ tự A → E.

In [ ]:
# 1. Clone public repository and import reusable project code
import gc
import importlib
import platform
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/ManhTanTran/Qaci-datascience.git"
REPO_BRANCH = "main"
REPO_COMMIT = None  # Pin a tested commit for a final reproducible run.
REPO_DIR = Path("/kaggle/working/Qaci-datascience")

def run_git(*arguments: str) -> None:
    subprocess.run(["git", "-C", str(REPO_DIR), *arguments], check=True)

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
if REPO_COMMIT is None:
    run_git("checkout", REPO_BRANCH)
    run_git("pull", "--ff-only", "origin", REPO_BRANCH)
else:
    run_git("fetch", "--depth", "1", "origin", REPO_COMMIT)
    run_git("checkout", "--detach", REPO_COMMIT)

GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
SRC_DIR = (REPO_DIR / "src").resolve()
sys.path.insert(0, str(SRC_DIR))
importlib.invalidate_caches()

from credit_scoring.artifacts import export_dataframe_artifact, export_json_artifact
from credit_scoring.data.home_credit import audit_home_credit_data, load_home_credit_data
from credit_scoring.evaluation.cross_validation import create_stratified_folds
from credit_scoring.experiments.home_credit_application import (
    prepare_application_data,
    resolve_e02_ablation_experiments,
)
from credit_scoring.modeling.lightgbm_model import run_lightgbm_cv
from credit_scoring.reproducibility import set_global_seed

set_global_seed(42)
print("Git commit:", GIT_COMMIT)
print("Python:", platform.python_version())

In [ ]:
# 2. Locked experiment configuration
RUN_MODES = {
    "smoke": {
        "sample_size": 5_000,
        "n_splits": 3,
        "n_estimators": 300,
        "early_stopping_rounds": 50,
    },
    "baseline": {
        "sample_size": None,
        "n_splits": 5,
        "n_estimators": 5_000,
        "early_stopping_rounds": 200,
    },
}
CONFIG = {
    "experiment_name": "E02_feature_family_ablation",
    "run_mode": "smoke",  # Change to baseline only after smoke passes.
    "data_dir": "/kaggle/input/competitions/home-credit-default-risk",
    "output_dir": "/kaggle/working/home_credit_outputs",
    "selected_experiments": None,  # None runs E01, A-E and E02-ALL.
    "random_state": 42,
}
if CONFIG["run_mode"] not in RUN_MODES:
    raise ValueError(f"Unknown run mode: {CONFIG['run_mode']}")
MODE = RUN_MODES[CONFIG["run_mode"]]
MODEL_CONFIG = {
    "learning_rate": 0.02,
    "n_estimators": MODE["n_estimators"],
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 80,
    "subsample": 0.8,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "random_state": CONFIG["random_state"],
    "n_jobs": -1,
    "verbosity": -1,
}
VALIDATION_CONFIG = {
    "n_splits": MODE["n_splits"],
    "shuffle": True,
    "random_state": CONFIG["random_state"],
    "early_stopping_rounds": MODE["early_stopping_rounds"],
    "keep_models": False,
}
EXPERIMENTS = resolve_e02_ablation_experiments(CONFIG["selected_experiments"])
OUTPUT_DIR = (
    Path(CONFIG["output_dir"])
    / CONFIG["experiment_name"]
    / CONFIG["run_mode"]
    / GIT_COMMIT[:8]
).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame([
    {"experiment": name, "families": ", ".join(families) or "E01 only"}
    for name, families in EXPERIMENTS.items()
]))

In [ ]:
# 3. Load only application_train and application_test
data = load_home_credit_data(
    CONFIG["data_dir"],
    tables=("application_train", "application_test"),
    nrows=MODE["sample_size"],
    reduce_memory=True,
    validate=True,
)
train = data["application_train"]
test = data["application_test"]
assert train["SK_ID_CURR"].is_unique
assert test["SK_ID_CURR"].is_unique
assert train["TARGET"].isin([0, 1]).all()
display(audit_home_credit_data(data))
print("Target rate:", float(train["TARGET"].mean()))

In [ ]:
# 4. Create one immutable fold assignment shared by every experiment
target = train["TARGET"].astype("int8")
folds = create_stratified_folds(
    target,
    n_splits=VALIDATION_CONFIG["n_splits"],
    shuffle=VALIDATION_CONFIG["shuffle"],
    random_state=VALIDATION_CONFIG["random_state"],
)
fold_assignment = np.full(len(train), -1, dtype=np.int8)
for fold_number, (_, valid_index) in enumerate(folds, start=1):
    fold_assignment[np.asarray(valid_index, dtype=int)] = fold_number
assert np.all(fold_assignment > 0)
assert len(np.unique(fold_assignment)) == VALIDATION_CONFIG["n_splits"]
print(pd.Series(fold_assignment).value_counts().sort_index())

In [ ]:
# 5. Run E01, each family, and E02-ALL sequentially
summary_rows = []
fold_rows = []
artifact_index = {}
baseline_fold_scores = None
baseline_oof_auc = None
baseline_columns = None
fold_fingerprint = None

for experiment_name, families in EXPERIMENTS.items():
    print(f"\nRunning {experiment_name}: {families or ('E01 only',)}")
    feature_set = "e01" if experiment_name == "E01_locked" else "e02"
    prepared = prepare_application_data(
        train,
        test,
        feature_set=feature_set,
        families=families or None,
    )
    result = run_lightgbm_cv(
        prepared.train_features,
        prepared.target,
        prepared.test_features,
        categorical_features=prepared.categorical_features,
        model_config=MODEL_CONFIG,
        validation_config=VALIDATION_CONFIG,
        folds=folds,
    )
    current_fingerprint = str(result["metadata"]["fold_fingerprint"])
    if fold_fingerprint is None:
        fold_fingerprint = current_fingerprint
    assert current_fingerprint == fold_fingerprint
    assert np.all(result["validation_counts"] == 1)
    assert np.isfinite(result["oof_predictions"]).all()
    assert np.isfinite(result["test_predictions"]).all()

    if experiment_name == "E01_locked":
        baseline_fold_scores = np.asarray(result["fold_scores"], dtype=float)
        baseline_oof_auc = float(result["oof_auc"])
        baseline_columns = list(prepared.train_features.columns)
    fold_scores = np.asarray(result["fold_scores"], dtype=float)
    fold_deltas = fold_scores - baseline_fold_scores
    added_features = [
        column for column in prepared.train_features.columns
        if column not in baseline_columns
    ]
    summary_row = {
        "experiment": experiment_name,
        "families": ",".join(families) or "E01 only",
        "n_features": prepared.train_features.shape[1],
        "n_added_features": len(added_features),
        "added_features": "|".join(added_features),
        "mean_fold_auc": float(result["mean_auc"]),
        "std_fold_auc": float(result["std_auc"]),
        "oof_auc": float(result["oof_auc"]),
        "delta_oof_auc_vs_e01": float(result["oof_auc"] - baseline_oof_auc),
        "positive_fold_count_vs_e01": int((fold_deltas > 0).sum()),
        "runtime_seconds": float(result["runtime"]),
        "fold_fingerprint": current_fingerprint,
    }
    for fold_number, (score, delta, best_iteration) in enumerate(
        zip(fold_scores, fold_deltas, result["best_iterations"], strict=True),
        start=1,
    ):
        summary_row[f"fold_{fold_number}_auc"] = float(score)
        summary_row[f"fold_{fold_number}_delta_vs_e01"] = float(delta)
        fold_rows.append({
            "experiment": experiment_name,
            "fold": fold_number,
            "auc": float(score),
            "delta_auc_vs_e01": float(delta),
            "best_iteration": int(best_iteration),
        })
    summary_rows.append(summary_row)

    experiment_dir = OUTPUT_DIR / "experiments" / experiment_name
    artifact_index[experiment_name] = {
        "oof_predictions": str(export_dataframe_artifact(pd.DataFrame({
            "SK_ID_CURR": prepared.train_ids.to_numpy(),
            "TARGET": prepared.target.to_numpy(),
            "FOLD": fold_assignment,
            "OOF_PREDICTION": result["oof_predictions"],
            "VALIDATION_COUNT": result["validation_counts"],
        }), experiment_dir / "oof_predictions.csv")),
        "test_predictions": str(export_dataframe_artifact(pd.DataFrame({
            "SK_ID_CURR": prepared.test_ids.to_numpy(),
            "TEST_PREDICTION": result["test_predictions"],
        }), experiment_dir / "test_predictions.csv")),
        "feature_importance": str(export_dataframe_artifact(
            result["feature_importance"], experiment_dir / "feature_importance.csv"
        )),
    }
    print(
        f"{experiment_name}: OOF={result['oof_auc']:.6f}, "
        f"delta={summary_row['delta_oof_auc_vs_e01']:+.6f}, "
        f"positive folds={summary_row['positive_fold_count_vs_e01']}/{len(folds)}"
    )
    result["fitted_models"].clear()
    del prepared, result
    gc.collect()

ablation_summary = pd.DataFrame(summary_rows)
fold_metrics = pd.DataFrame(fold_rows)
display(ablation_summary.sort_values("oof_auc", ascending=False))

In [ ]:
# 6. Export summary and reproducibility metadata
def installed_version(package_name: str) -> str | None:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None

summary_path = export_dataframe_artifact(ablation_summary, OUTPUT_DIR / "ablation_summary.csv")
fold_path = export_dataframe_artifact(fold_metrics, OUTPUT_DIR / "fold_metrics.csv")
config_path = export_json_artifact({
    "config": CONFIG,
    "mode": MODE,
    "model_config": MODEL_CONFIG,
    "validation_config": VALIDATION_CONFIG,
    "experiments": EXPERIMENTS,
}, OUTPUT_DIR / "config.json")
metadata_path = export_json_artifact({
    "status": "completed",
    "git_commit": GIT_COMMIT,
    "dataset_path": str(Path(CONFIG["data_dir"]).resolve()),
    "n_train": len(train),
    "n_test": len(test),
    "fold_fingerprint": fold_fingerprint,
    "environment": {
        "python": platform.python_version(),
        "packages": {name: installed_version(name) for name in [
            "numpy", "pandas", "scikit-learn", "lightgbm"
        ]},
    },
    "summary_path": str(summary_path),
    "fold_metrics_path": str(fold_path),
    "config_path": str(config_path),
    "experiment_artifacts": artifact_index,
}, OUTPUT_DIR / "run_metadata.json")
print("Output directory:", OUTPUT_DIR)
print("Run metadata:", metadata_path)

In [ ]:
# 7. Compare measured gains; this plot is generated only from actual run results
plot_data = ablation_summary.set_index("experiment")["delta_oof_auc_vs_e01"].drop("E01_locked")
colors = ["#15803d" if value > 0 else "#b91c1c" for value in plot_data]
ax = plot_data.plot(kind="barh", figsize=(9, 5), color=colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("E02 feature-family ablation — Δ OOF AUC vs E01")
ax.set_xlabel("Δ OOF AUC")
ax.set_ylabel("Experiment")
plt.tight_layout()
plt.show()

display(ablation_summary[[
    "experiment", "n_added_features", "oof_auc",
    "delta_oof_auc_vs_e01", "positive_fold_count_vs_e01",
    "std_fold_auc", "runtime_seconds",
]].sort_values("delta_oof_auc_vs_e01", ascending=False))

## Cách quyết định

Một family chỉ nên được giữ khi `delta_oof_auc_vs_e01 > 0`, gain xuất hiện ở nhiều fold và độ lệch fold không xấu đi rõ rệt. Không kết luận từ feature importance hoặc public leaderboard.

Sau full baseline run, tải toàn bộ thư mục output và cập nhật `docs/experiments/experiment_log.md` bằng metric thực tế. Nếu không chạy full data thì không ghi smoke metric như kết quả competition.